# Product ABC Analysis ETL

## Purpose
Provides ABC classification of products based on revenue contribution using the Pareto principle (80/20 rule). This materialized table enables:
* Identify high-value products (Class A: top 80% revenue)
* Prioritize inventory management
* Focus marketing efforts on key products
* Support strategic product decisions
* **Full dataset access** (49,685 products) without view visualization limits

## Input → Output
* **Source:** `big_data.gold.product_performance`
* **Target:** `big_data.gold.product_abc_analysis`
* **Primary Key:** `product_id`

## Transformations
1. Load product performance metrics from gold layer
2. Calculate total revenue across all products
3. Compute cumulative revenue for each product (ordered by revenue DESC)
4. Calculate cumulative revenue percentage
5. Classify products into ABC classes:
   * **Class A:** Top products contributing to first 80% of revenue
   * **Class B:** Products contributing from 80% to 95% of revenue
   * **Class C:** Remaining products contributing last 5% of revenue
6. Add metadata timestamp

## Data Quality
* **Technical:** NOT NULL (PK), UNIQUE (PK), critical columns validation (estimated_revenue_usd)
* **Business:** ABC class distribution checks (A: ~9%, B: ~19%, C: ~71%)

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    IntegerType, 
    LongType,
    DoubleType, 
    DecimalType, 
    StringType,
    TimestampType
)

In [0]:
# Schema configuration
source_schema = "big_data.gold"
target_schema = "big_data.gold"

# Source Tables (full paths)
product_performance = f"{source_schema}.product_performance"

source_tables = [
    product_performance
]

# Target Table (full path)
target_table = f"{target_schema}.product_abc_analysis"

# Primary Key columns for validation
primary_key_columns = ["product_id"]

# Critical columns (NOT NULL required)
critical_columns = ["product_name", "estimated_revenue_usd", "abc_class"]

# Print configuration
print("Configuration:")
print(f"  Source_tables: {source_tables}")
print(f"  Target: {target_table}")
print(f"  Primary Key: {primary_key_columns}")

### TRANSFORMATION

In [0]:
print("Step 1: Loading product performance and calculating total revenue...")

# Load product performance data
df_products = spark.table(product_performance) \
    .select(
        F.col("product_id").cast(IntegerType()),
        F.col("product_name"),
        F.col("department"),
        F.col("times_ordered").cast(LongType()),
        F.col("estimated_revenue_usd").cast(DecimalType(21, 2)),
        F.col("reorder_rate").cast(DoubleType())
    ) \
    .filter(F.col("product_id").isNotNull()) \
    .filter(F.col("estimated_revenue_usd").isNotNull())

product_count = df_products.count()
print(f"  Loaded: {product_count:,} products")

# Calculate total revenue across all products
total_revenue = df_products.agg(F.sum("estimated_revenue_usd").alias("total")).collect()[0]["total"]
print(f"  Total revenue: ${total_revenue:,.2f}")

In [0]:
print("Step 2: Calculating cumulative revenue...")

# Define window for cumulative sum (ordered by revenue DESC)
window_spec = Window.orderBy(F.col("estimated_revenue_usd").desc()) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate cumulative revenue
df_cumulative = df_products \
    .withColumn(
        "cumulative_revenue",
        F.sum("estimated_revenue_usd").over(window_spec)
    ) \
    .withColumn(
        "total_revenue",
        F.lit(total_revenue)
    ) \
    .withColumn(
        "cumulative_revenue_pct",
        F.round((F.col("cumulative_revenue") / F.col("total_revenue")) * 100, 2)
    )

print(f"  Cumulative revenue calculated for {df_cumulative.count():,} products")

In [0]:
print("Step 3: Classifying products into ABC classes...")

# Classify products based on cumulative revenue percentage
df_classified = df_cumulative \
    .withColumn(
        "abc_class",
        F.when(F.col("cumulative_revenue_pct") <= 80, "A")
         .when(F.col("cumulative_revenue_pct") <= 95, "B")
         .otherwise("C")
    ) \
    .drop("total_revenue")

# Show distribution
class_distribution = df_classified.groupBy("abc_class") \
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / F.lit(product_count)) * 100, 2).alias("pct")
    ) \
    .orderBy("abc_class") \
    .collect()

print("  ABC Class Distribution:")
for row in class_distribution:
    print(f"    Class {row['abc_class']}: {row['count']:,} products ({row['pct']}%)")

In [0]:
print("Step 4: Adding metadata timestamp...")

# Add gold timestamp
df_result = df_classified \
    .withColumn("_gold_timestamp", F.current_timestamp())

print(f"  Final dataset: {df_result.count():,} rows")
print("  Transformation complete!")

### DATA QUALITY

In [0]:
# Execute technical validations using UTILS orchestrator
validation_technical, total_rows = technical_validations(
    df=df_result,
    primary_key_columns=primary_key_columns,
    critical_columns=critical_columns,
    range_checks=[]  # No range checks needed for this table
)

In [0]:
# Business validations
validation_business = True

print("\n" + "="*60)
print("BUSINESS VALIDATIONS")
print("="*60)

# Check 1: ABC class distribution (expected: A ~9%, B ~19%, C ~71%)
class_counts = df_result.groupBy("abc_class").count().collect()
class_dict = {row['abc_class']: row['count'] for row in class_counts}
total = sum(class_dict.values())

for cls in ['A', 'B', 'C']:
    count = class_dict.get(cls, 0)
    pct = (count / total) * 100
    print(f"  Class {cls}: {count:,} products ({pct:.2f}%)")

# Validate expected distributions (with tolerance)
class_a_pct = (class_dict.get('A', 0) / total) * 100
class_b_pct = (class_dict.get('B', 0) / total) * 100
class_c_pct = (class_dict.get('C', 0) / total) * 100

if not (5 <= class_a_pct <= 15):  # Expected ~9%
    print(f"  ⚠️ Class A distribution outside expected range: {class_a_pct:.2f}%")
    validation_business = False

if not (10 <= class_b_pct <= 30):  # Expected ~19%
    print(f"  ⚠️ Class B distribution outside expected range: {class_b_pct:.2f}%")
    validation_business = False

if not (60 <= class_c_pct <= 80):  # Expected ~71%
    print(f"  ⚠️ Class C distribution outside expected range: {class_c_pct:.2f}%")
    validation_business = False

# Check 2: Cumulative revenue percentage should reach ~100%
max_cumulative_pct = df_result.agg(F.max("cumulative_revenue_pct")).collect()[0][0]
if max_cumulative_pct < 99 or max_cumulative_pct > 101:
    print(f"  ⚠️ Max cumulative revenue % is {max_cumulative_pct}% (expected ~100%)")
    validation_business = False
else:
    print(f"  ✓ Cumulative revenue reaches {max_cumulative_pct}%")

if validation_business:
    print("\n✓ All business validations passed")
else:
    print("\n✗ Business validation failed")

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, target_table)
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")

In [0]:
%sql
-- Query the materialized Delta table
-- Shows ALL 49,685 products without visualization limits
-- Ordered by revenue descending

SELECT 
  product_id,
  product_name,
  department,
  times_ordered,
  estimated_revenue_usd,
  reorder_rate,
  cumulative_revenue_pct,
  abc_class
FROM big_data.gold.product_abc_analysis
ORDER BY estimated_revenue_usd DESC;

In [0]:
%sql
-- Filter to show ONLY Class C products from the materialized table
-- These are the 35,517 long-tail products contributing the last 5% of revenue
-- Now you can see ALL of them without the 10k view limit!

SELECT 
  product_id,
  product_name,
  department,
  times_ordered,
  estimated_revenue_usd,
  reorder_rate,
  cumulative_revenue_pct,
  abc_class
FROM big_data.gold.product_abc_analysis
WHERE abc_class = 'C'
ORDER BY estimated_revenue_usd DESC;

In [0]:
%sql
-- Summary statistics showing the complete materialized table
-- Confirms all 49,685 products are persisted with correct ABC distribution

SELECT 
  abc_class,
  COUNT(*) as product_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct_of_products,
  ROUND(MIN(estimated_revenue_usd), 2) as min_revenue,
  ROUND(MAX(estimated_revenue_usd), 2) as max_revenue,
  ROUND(AVG(estimated_revenue_usd), 2) as avg_revenue,
  ROUND(SUM(estimated_revenue_usd), 2) as total_revenue
FROM big_data.gold.product_abc_analysis
GROUP BY abc_class
ORDER BY abc_class;